# Sistema de Perguntas e Respostas Extrativo em Português com BERTimbau

**Disciplina:** Processamento de Linguagem Natural  
**Autores:** Alexandre da Fonseca e Riquelmy Henrique Silva  
**Professor:** Leonardo de Lellis Rossi  
**Instituição:** FATEC  
**Data:** Maio/2026

---

Este notebook implementa um sistema de **Perguntas e Respostas Extrativo (Extractive QA)** em português utilizando o modelo **BERTimbau** (`pierreguillou/bert-base-cased-squad-v1.1-portuguese`), aplicado a um corpus de ~40.000 palavras extraído da Wikipédia em português sobre Inteligência Artificial e Ciência de Dados.

**Fluxo do sistema:**
```
Pergunta → Recuperação TF-IDF → Contexto relevante → BERTimbau → Resposta extraída
```

## 1. Instalação e Configuração

> Execute esta célula apenas uma vez. O Colab pode solicitar reinício do ambiente após a instalação.

In [ ]:
!pip install wikipedia-api transformers torch scikit-learn pandas matplotlib seaborn -q
print('Instalacao concluida!')

In [ ]:
import wikipediaapi
import re
import string
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline
import torch
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid', palette='muted')

device = 0 if torch.cuda.is_available() else -1
print(f'Dispositivo: {"GPU (CUDA)" if device == 0 else "CPU"}')
print('Bibliotecas carregadas com sucesso!')

## 2. Carregamento do Corpus

O corpus é formado por artigos da **Wikipédia em português** sobre temas de Inteligência Artificial e Ciência de Dados, totalizando aproximadamente **40.000 palavras**.

In [ ]:
wiki = wikipediaapi.Wikipedia(
    user_agent='TrabalhoFinalPLN/1.0 (alefonsecabb@gmail.com)',
    language='pt',
    extract_format=wikipediaapi.ExtractFormat.WIKI
)

ARTIGOS = [
    'Inteligência artificial',
    'Aprendizado de máquina',
    'Aprendizado profundo',
    'Rede neural artificial',
    'Processamento de linguagem natural',
    'BERT',
    'Ciência de dados',
    'Mineração de dados',
    'Visão computacional',
    'Robótica'
]

def baixar_artigo(titulo):
    pagina = wiki.page(titulo)
    if pagina.exists() and len(pagina.text) > 200:
        return pagina.text
    return ''

corpus_textos = {}
corpus_completo = ''

for artigo in ARTIGOS:
    texto = baixar_artigo(artigo)
    if texto:
        corpus_textos[artigo] = texto
        corpus_completo += '\n\n' + texto
        print(f'OK  {artigo}: {len(texto.split()):,} palavras')
    else:
        print(f'AVISO: "{artigo}" nao encontrado na Wikipedia PT')

total = len(corpus_completo.split())
print(f'\nTotal do corpus: {total:,} palavras')
print(f'Total de caracteres: {len(corpus_completo):,}')

In [ ]:
# Estatisticas e visualizacao do corpus
palavras_por_artigo = {k: len(v.split()) for k, v in corpus_textos.items()}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Grafico de barras
cores = sns.color_palette('Blues_d', len(palavras_por_artigo))
bars = axes[0].barh(
    list(palavras_por_artigo.keys()),
    list(palavras_por_artigo.values()),
    color=cores
)
axes[0].set_xlabel('Numero de Palavras')
axes[0].set_title('Distribuicao de Palavras por Artigo')
for bar, val in zip(bars, palavras_por_artigo.values()):
    axes[0].text(bar.get_width() + 50, bar.get_y() + bar.get_height()/2,
                 f'{val:,}', va='center', fontsize=8)

# Grafico de pizza
axes[1].pie(
    list(palavras_por_artigo.values()),
    labels=list(palavras_por_artigo.keys()),
    autopct='%1.1f%%',
    startangle=90,
    textprops={'fontsize': 7}
)
axes[1].set_title('Proporcao por Artigo')

plt.suptitle(f'Corpus: {sum(palavras_por_artigo.values()):,} palavras no total', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\nEstatisticas do Corpus:')
print(f'  Palavras:    {len(corpus_completo.split()):>8,}')
print(f'  Caracteres:  {len(corpus_completo):>8,}')
linhas = [l for l in corpus_completo.split('\n') if l.strip()]
print(f'  Paragrafos:  {len(linhas):>8,}')

## 3. Pré-processamento do Texto

O texto é limpo e dividido em **chunks de contexto** (blocos de até 400 palavras). Um recuperador **TF-IDF** seleciona os chunks mais relevantes para cada pergunta.

In [ ]:
def limpar_texto(texto):
    texto = re.sub(r'==+[^=]+==+', '', texto)   # remove cabecalhos wiki
    texto = re.sub(r'\n{3,}', '\n\n', texto)     # normaliza quebras de linha
    texto = re.sub(r'[ \t]{2,}', ' ', texto)     # normaliza espacos
    return texto.strip()

def criar_chunks(texto, max_palavras=400, sobreposicao=50):
    paragrafos = [p.strip() for p in texto.split('\n') if len(p.strip()) > 30]
    chunks = []
    buffer = []
    contagem = 0
    for p in paragrafos:
        palavras_p = p.split()
        if contagem + len(palavras_p) > max_palavras and buffer:
            chunks.append(' '.join(buffer))
            # sobreposicao: mantém ultimas frases
            buffer = buffer[-sobreposicao:] if len(buffer) > sobreposicao else []
            contagem = sum(len(b.split()) for b in buffer)
        buffer.append(p)
        contagem += len(palavras_p)
    if buffer:
        chunks.append(' '.join(buffer))
    return chunks

corpus_limpo = limpar_texto(corpus_completo)
chunks = criar_chunks(corpus_limpo)

tamanhos = [len(c.split()) for c in chunks]
print(f'Chunks criados:       {len(chunks)}')
print(f'Tamanho medio:        {np.mean(tamanhos):.0f} palavras')
print(f'Tamanho maximo:       {max(tamanhos)} palavras')
print(f'Tamanho minimo:       {min(tamanhos)} palavras')
print(f'\nExemplo de chunk #0:')
print(chunks[0][:400] + '...')

In [ ]:
# Vetorizador TF-IDF para recuperacao de contexto
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=60000,
    sublinear_tf=True
)
tfidf_matrix = vectorizer.fit_transform(chunks)

def buscar_contexto(pergunta, top_k=3):
    vec = vectorizer.transform([pergunta])
    scores = cosine_similarity(vec, tfidf_matrix).flatten()
    top_idx = scores.argsort()[-top_k:][::-1]
    contexto = ' '.join([chunks[i] for i in top_idx])
    # limitar a 450 palavras para caber no BERT (512 tokens)
    palavras = contexto.split()
    if len(palavras) > 450:
        contexto = ' '.join(palavras[:450])
    return contexto

print('Recuperador TF-IDF pronto!')
print(f'Vocabulario: {len(vectorizer.vocabulary_):,} termos')

# Teste rapido
ctx_teste = buscar_contexto('O que e BERT?')
print(f'\nTeste - contexto recuperado para "O que e BERT?" ({len(ctx_teste.split())} palavras):')
print(ctx_teste[:300] + '...')

## 4. Carregamento do Modelo BERTimbau

Utilizamos o modelo **`pierreguillou/bert-base-cased-squad-v1.1-portuguese`**, baseado no BERTimbau e ajustado (*fine-tuned*) no dataset SQuAD v1.1 traduzido para o português.

> A primeira execução baixa ~400 MB do HuggingFace Hub. Com GPU, o processamento é ~10x mais rápido.

In [ ]:
MODEL_NAME = 'pierreguillou/bert-base-cased-squad-v1.1-portuguese'

print(f'Carregando modelo: {MODEL_NAME}')
print('Aguarde...')

qa_pipeline = pipeline(
    'question-answering',
    model=MODEL_NAME,
    tokenizer=MODEL_NAME,
    device=device
)

print(f'Modelo carregado com sucesso!')
print(f'Dispositivo em uso: {"GPU" if device == 0 else "CPU"}')

In [ ]:
def responder(pergunta, contexto=None, top_k_chunks=3, pipeline_model=None):
    if pipeline_model is None:
        pipeline_model = qa_pipeline  # BERTimbau por padrão
    if contexto is None:
        contexto = buscar_contexto(pergunta, top_k=top_k_chunks)
    resultado = pipeline_model(question=pergunta, context=contexto)
    return {
        'pergunta': pergunta,
        'resposta': resultado['answer'],
        'score': resultado['score'],
        'start': resultado['start'],
        'end': resultado['end'],
        'contexto': contexto
    }

# Teste inicial com corpus
perguntas_teste = [
    'O que e inteligencia artificial?',
    'O que significa a sigla BERT?',
    'Quais sao os tipos de aprendizado de maquina?'
]

print('=== TESTE INICIAL DO SISTEMA ===')
for p in perguntas_teste:
    r = responder(p)
    print(f'\nPergunta: {p}')
    print(f'Resposta: {r["resposta"]}')
    print(f'Score:    {r["score"]:.2%}')
    print('-' * 50)


## 5. Interface Interativa de Perguntas

Demonstração do sistema respondendo perguntas sobre o corpus.

In [ ]:
perguntas_demo = [
    'Quem cunhou o termo inteligencia artificial?',
    'Em que ano foi realizada a Conferencia de Dartmouth?',
    'O que e aprendizado profundo?',
    'O que sao redes neurais convolucionais?',
    'Quem propôs o perceptron?',
    'O que e mineracao de dados?',
    'O que e visao computacional?',
    'O que e processamento de linguagem natural?'
]

print('=' * 65)
print('   SISTEMA DE PERGUNTAS E RESPOSTAS - WIKIPEDIA PT-BR')
print('=' * 65)

for i, pergunta in enumerate(perguntas_demo, 1):
    resultado = responder(pergunta)
    status = 'ALTA' if resultado['score'] > 0.7 else ('MEDIA' if resultado['score'] > 0.4 else 'BAIXA')
    print(f'\n[{i:02d}] Pergunta : {pergunta}')
    print(f'     Resposta : {resultado["resposta"]}')
    print(f'     Confianca: {resultado["score"]:.2%} ({status})')

## 6. Dataset de Avaliação (30 Pares QA)

Conjunto de 30 pares (pergunta, resposta esperada, contexto) criados manualmente, cobrindo 6 categorias temáticas com 5 tipos de perguntas: definição, factual-pessoa, factual-data, factual-local e enumeração.

In [ ]:
dataset_qa = [
    # ── INTELIGÊNCIA ARTIFICIAL ────────────────────────────────────────────
    {
        'categoria': 'Inteligência Artificial',
        'tipo': 'Factual - Pessoa',
        'pergunta': 'Quem cunhou o termo inteligência artificial?',
        'resposta_esperada': 'John McCarthy',
        'contexto': 'O termo inteligência artificial foi cunhado por John McCarthy em 1956, durante a Conferência de Dartmouth, realizada no Dartmouth College, nos Estados Unidos. McCarthy, considerado um dos pais da inteligência artificial, também criou a linguagem de programação Lisp e contribuiu de forma decisiva para o estabelecimento da área como disciplina científica.'
    },
    {
        'categoria': 'Inteligência Artificial',
        'tipo': 'Factual - Data',
        'pergunta': 'Em que ano foi realizada a Conferência de Dartmouth?',
        'resposta_esperada': '1956',
        'contexto': 'A Conferência de Dartmouth foi realizada em 1956 no Dartmouth College, em Hanover, New Hampshire, nos Estados Unidos. Organizada por John McCarthy, Marvin Minsky, Nathaniel Rochester e Claude Shannon, o evento é considerado o marco fundador da inteligência artificial como disciplina científica independente.'
    },
    {
        'categoria': 'Inteligência Artificial',
        'tipo': 'Factual - Pessoa',
        'pergunta': 'Quem propôs o Teste de Turing?',
        'resposta_esperada': 'Alan Turing',
        'contexto': 'O Teste de Turing foi proposto por Alan Turing em 1950, no artigo Computing Machinery and Intelligence, publicado na revista Mind. O teste consiste em um avaliador humano conversar por escrito com um humano e com uma máquina sem saber qual é qual. Se o avaliador não conseguir distinguir a máquina do humano, considera-se que a máquina demonstrou comportamento inteligente.'
    },
    {
        'categoria': 'Inteligência Artificial',
        'tipo': 'Factual - Data',
        'pergunta': 'Em que ano Alan Turing publicou o artigo sobre o teste que leva seu nome?',
        'resposta_esperada': '1950',
        'contexto': 'O Teste de Turing foi proposto por Alan Turing em 1950, no artigo Computing Machinery and Intelligence, publicado na revista Mind. O teste consiste em um avaliador humano conversar por escrito com um humano e com uma máquina sem saber qual é qual.'
    },
    {
        'categoria': 'Inteligência Artificial',
        'tipo': 'Definição',
        'pergunta': 'O que são sistemas especialistas em inteligência artificial?',
        'resposta_esperada': 'programas de computador que simulam o conhecimento e o raciocínio de um especialista humano em um domínio específico',
        'contexto': 'Os sistemas especialistas são programas de computador que simulam o conhecimento e o raciocínio de um especialista humano em um domínio específico. Desenvolvidos principalmente nas décadas de 1970 e 1980, foram uma das primeiras aplicações práticas bem-sucedidas da inteligência artificial. Exemplos incluem o MYCIN, usado para diagnóstico de infecções bacterianas, e o DENDRAL, para análise de estruturas químicas.'
    },
    # ── APRENDIZADO DE MÁQUINA ─────────────────────────────────────────────
    {
        'categoria': 'Aprendizado de Máquina',
        'tipo': 'Factual - Pessoa',
        'pergunta': 'Quem introduziu o conceito de aprendizado de máquina?',
        'resposta_esperada': 'Arthur Samuel',
        'contexto': 'O conceito de aprendizado de máquina foi introduzido por Arthur Samuel em 1959, que o definiu como o campo de estudo que confere aos computadores a capacidade de aprender sem serem explicitamente programados. Samuel desenvolveu um dos primeiros programas de aprendizado de máquina: um jogo de damas capaz de melhorar sua performance por meio da experiência acumulada.'
    },
    {
        'categoria': 'Aprendizado de Máquina',
        'tipo': 'Factual - Data',
        'pergunta': 'Em que ano Arthur Samuel introduziu o conceito de aprendizado de máquina?',
        'resposta_esperada': '1959',
        'contexto': 'O conceito de aprendizado de máquina foi introduzido por Arthur Samuel em 1959, que o definiu como o campo de estudo que confere aos computadores a capacidade de aprender sem serem explicitamente programados. Samuel desenvolveu um dos primeiros programas de aprendizado de máquina: um jogo de damas capaz de melhorar sua performance por meio da experiência acumulada.'
    },
    {
        'categoria': 'Aprendizado de Máquina',
        'tipo': 'Enumeração',
        'pergunta': 'Quais são os três paradigmas principais do aprendizado de máquina?',
        'resposta_esperada': 'aprendizado supervisionado, aprendizado não supervisionado e aprendizado por reforço',
        'contexto': 'O aprendizado de máquina é dividido em três paradigmas principais: aprendizado supervisionado, aprendizado não supervisionado e aprendizado por reforço. No aprendizado supervisionado, os algoritmos são treinados com exemplos rotulados. No aprendizado não supervisionado, o algoritmo busca padrões em dados sem rótulos. No aprendizado por reforço, um agente aprende por meio de interações com o ambiente, recebendo recompensas ou penalidades.'
    },
    {
        'categoria': 'Aprendizado de Máquina',
        'tipo': 'Definição',
        'pergunta': 'O que é overfitting em aprendizado de máquina?',
        'resposta_esperada': 'um problema que ocorre quando um modelo aprende os dados de treinamento com muita precisão, incluindo o ruído e os detalhes irrelevantes',
        'contexto': 'O overfitting, ou sobreajuste, é um problema que ocorre quando um modelo aprende os dados de treinamento com muita precisão, incluindo o ruído e os detalhes irrelevantes, perdendo a capacidade de generalizar para novos dados. Um modelo com overfitting apresenta alta acurácia no conjunto de treinamento, porém baixo desempenho no conjunto de teste ou em dados do mundo real.'
    },
    {
        'categoria': 'Aprendizado de Máquina',
        'tipo': 'Definição',
        'pergunta': 'O que é aprendizado por reforço?',
        'resposta_esperada': 'um paradigma de aprendizado em que um agente aprende por meio de interações com o ambiente, recebendo recompensas ou penalidades por suas ações',
        'contexto': 'O aprendizado por reforço é um paradigma de aprendizado em que um agente aprende por meio de interações com o ambiente, recebendo recompensas ou penalidades por suas ações. O objetivo do agente é maximizar a recompensa acumulada ao longo do tempo. Esse paradigma foi inspirado na psicologia comportamental e tem aplicações em jogos, robótica e sistemas de controle autônomo.'
    },
    # ── APRENDIZADO PROFUNDO ───────────────────────────────────────────────
    {
        'categoria': 'Aprendizado Profundo',
        'tipo': 'Definição',
        'pergunta': 'O que é aprendizado profundo?',
        'resposta_esperada': 'uma subárea do aprendizado de máquina que utiliza redes neurais artificiais com múltiplas camadas para aprender representações de dados em diferentes níveis de abstração',
        'contexto': 'O aprendizado profundo é uma subárea do aprendizado de máquina que utiliza redes neurais artificiais com múltiplas camadas para aprender representações de dados em diferentes níveis de abstração. O termo profundo refere-se ao número de camadas ocultas na rede neural. O aprendizado profundo revolucionou áreas como reconhecimento de imagens, síntese de fala e processamento de linguagem natural.'
    },
    {
        'categoria': 'Aprendizado Profundo',
        'tipo': 'Definição',
        'pergunta': 'O que são redes neurais convolucionais?',
        'resposta_esperada': 'um tipo de rede neural profunda especialmente projetada para processar dados com estrutura em grade, como imagens',
        'contexto': 'As redes neurais convolucionais (CNN, do inglês Convolutional Neural Networks) são um tipo de rede neural profunda especialmente projetada para processar dados com estrutura em grade, como imagens. Utilizam operações de convolução para extrair características locais dos dados, como bordas, texturas e padrões. As CNNs são amplamente utilizadas em classificação de imagens, detecção de objetos e reconhecimento facial.'
    },
    {
        'categoria': 'Aprendizado Profundo',
        'tipo': 'Definição',
        'pergunta': 'O que são redes neurais recorrentes?',
        'resposta_esperada': 'uma classe de redes neurais em que as conexões entre os nós formam um grafo orientado ao longo de uma sequência temporal',
        'contexto': 'As redes neurais recorrentes (RNN, do inglês Recurrent Neural Networks) são uma classe de redes neurais em que as conexões entre os nós formam um grafo orientado ao longo de uma sequência temporal. Diferentemente das redes feedforward, as RNNs utilizam seu estado interno como memória para processar sequências de dados. São amplamente usadas em tarefas de PLN e reconhecimento de fala.'
    },
    {
        'categoria': 'Aprendizado Profundo',
        'tipo': 'Definição',
        'pergunta': 'O que é transferência de aprendizado em aprendizado profundo?',
        'resposta_esperada': 'uma técnica que consiste em reutilizar um modelo pré-treinado em uma tarefa como ponto de partida para um modelo em uma tarefa diferente',
        'contexto': 'A transferência de aprendizado é uma técnica que consiste em reutilizar um modelo pré-treinado em uma tarefa como ponto de partida para um modelo em uma tarefa diferente. Em vez de treinar um modelo do zero, o conhecimento adquirido em uma tarefa com grandes volumes de dados é transferido para uma nova tarefa com menos dados disponíveis. Essa abordagem reduziu drasticamente o tempo e os recursos necessários para treinar modelos de PLN.'
    },
    # ── REDES NEURAIS ─────────────────────────────────────────────────────
    {
        'categoria': 'Redes Neurais',
        'tipo': 'Factual - Pessoa',
        'pergunta': 'Quem propôs o perceptron?',
        'resposta_esperada': 'Frank Rosenblatt',
        'contexto': 'O perceptron é o modelo mais simples de rede neural artificial, proposto por Frank Rosenblatt em 1957. Trata-se de um classificador binário linear que recebe entradas numéricas, aplica pesos a cada entrada e usa uma função de ativação para produzir uma saída. A ideia foi inspirada no funcionamento dos neurônios biológicos e representou o início do campo das redes neurais artificiais.'
    },
    {
        'categoria': 'Redes Neurais',
        'tipo': 'Factual - Data',
        'pergunta': 'Em que ano Frank Rosenblatt propôs o perceptron?',
        'resposta_esperada': '1957',
        'contexto': 'O perceptron é o modelo mais simples de rede neural artificial, proposto por Frank Rosenblatt em 1957. Trata-se de um classificador binário linear que recebe entradas numéricas, aplica pesos a cada entrada e usa uma função de ativação para produzir uma saída.'
    },
    {
        'categoria': 'Redes Neurais',
        'tipo': 'Definição',
        'pergunta': 'O que é o algoritmo de backpropagation?',
        'resposta_esperada': 'o principal algoritmo usado para treinar redes neurais artificiais, que calcula o gradiente da função de perda em relação a cada peso da rede propagando o erro da saída para a entrada',
        'contexto': 'O backpropagation, ou retropropagação do erro, é o principal algoritmo usado para treinar redes neurais artificiais, que calcula o gradiente da função de perda em relação a cada peso da rede propagando o erro da saída para a entrada. O algoritmo foi popularizado por David Rumelhart, Geoffrey Hinton e Ronald Williams em 1986 e tornou possível o treinamento eficiente de redes com múltiplas camadas ocultas.'
    },
    {
        'categoria': 'Redes Neurais',
        'tipo': 'Factual - Data',
        'pergunta': 'Em que ano Rumelhart, Hinton e Williams popularizaram o backpropagation?',
        'resposta_esperada': '1986',
        'contexto': 'O backpropagation, ou retropropagação do erro, é o principal algoritmo usado para treinar redes neurais artificiais. O algoritmo foi popularizado por David Rumelhart, Geoffrey Hinton e Ronald Williams em 1986 e tornou possível o treinamento eficiente de redes com múltiplas camadas ocultas.'
    },
    # ── PLN E BERT ────────────────────────────────────────────────────────
    {
        'categoria': 'PLN e BERT',
        'tipo': 'Definição',
        'pergunta': 'O que é processamento de linguagem natural?',
        'resposta_esperada': 'uma subárea da inteligência artificial e da linguística computacional que estuda como os computadores podem processar e analisar grandes quantidades de dados em linguagem natural',
        'contexto': 'O processamento de linguagem natural (PLN) é uma subárea da inteligência artificial e da linguística computacional que estuda como os computadores podem processar e analisar grandes quantidades de dados em linguagem natural. O PLN abrange tarefas como análise de sentimentos, tradução automática, reconhecimento de entidades nomeadas, sumarização de textos e sistemas de perguntas e respostas.'
    },
    {
        'categoria': 'PLN e BERT',
        'tipo': 'Definição',
        'pergunta': 'O que é tokenização em PLN?',
        'resposta_esperada': 'o processo de dividir um texto em unidades menores chamadas tokens, que podem ser palavras, subpalavras ou caracteres',
        'contexto': 'A tokenização é o processo de dividir um texto em unidades menores chamadas tokens, que podem ser palavras, subpalavras ou caracteres. É geralmente o primeiro passo no pré-processamento de texto em sistemas de PLN. A tokenização permite que os modelos processem o texto de forma estruturada, transformando sequências de caracteres em sequências de tokens numericamente representáveis.'
    },
    {
        'categoria': 'PLN e BERT',
        'tipo': 'Factual - Sigla',
        'pergunta': 'O que significa a sigla BERT?',
        'resposta_esperada': 'Bidirectional Encoder Representations from Transformers',
        'contexto': 'BERT, sigla para Bidirectional Encoder Representations from Transformers, é um modelo de linguagem baseado na arquitetura Transformer desenvolvido e publicado pela equipe do Google AI Language em 2018. O BERT revolucionou o campo do processamento de linguagem natural ao introduzir o pré-treinamento bidirecional, permitindo que o modelo considere o contexto tanto à esquerda quanto à direita de cada token durante o pré-treinamento.'
    },
    {
        'categoria': 'PLN e BERT',
        'tipo': 'Factual - Empresa',
        'pergunta': 'Qual empresa desenvolveu o modelo BERT?',
        'resposta_esperada': 'Google',
        'contexto': 'BERT, sigla para Bidirectional Encoder Representations from Transformers, é um modelo de linguagem baseado na arquitetura Transformer desenvolvido e publicado pela equipe do Google AI Language em 2018. O BERT revolucionou o campo do processamento de linguagem natural ao introduzir o pré-treinamento bidirecional.'
    },
    {
        'categoria': 'PLN e BERT',
        'tipo': 'Factual - Data',
        'pergunta': 'Em que ano o modelo BERT foi publicado?',
        'resposta_esperada': '2018',
        'contexto': 'BERT, sigla para Bidirectional Encoder Representations from Transformers, é um modelo de linguagem baseado na arquitetura Transformer desenvolvido e publicado pela equipe do Google AI Language em 2018. O BERT revolucionou o campo do processamento de linguagem natural ao introduzir o pré-treinamento bidirecional.'
    },
    {
        'categoria': 'PLN e BERT',
        'tipo': 'Definição',
        'pergunta': 'O que é o BERTimbau?',
        'resposta_esperada': 'um modelo de linguagem pré-treinado baseado na arquitetura BERT desenvolvido para o idioma português',
        'contexto': 'O BERTimbau é um modelo de linguagem pré-treinado baseado na arquitetura BERT desenvolvido para o idioma português. Foi criado por pesquisadores do NeuralMind e publicado em 2020. O BERTimbau foi treinado em um corpus de textos em português com mais de 2,68 bilhões de palavras, extraídos da Wikipédia em português e do corpus brWaC (Brazilian Web as Corpus).'
    },
    {
        'categoria': 'PLN e BERT',
        'tipo': 'Definição',
        'pergunta': 'O que é um sistema de perguntas e respostas?',
        'resposta_esperada': 'um sistema de processamento de linguagem natural que responde automaticamente a perguntas formuladas em linguagem natural',
        'contexto': 'Um sistema de perguntas e respostas (QA, do inglês Question Answering) é um sistema de processamento de linguagem natural que responde automaticamente a perguntas formuladas em linguagem natural. Existem dois tipos principais: sistemas extrativos, que localizam e extraem a resposta de um texto dado, e sistemas generativos, que geram uma resposta com base no conhecimento adquirido durante o treinamento.'
    },
    {
        'categoria': 'PLN e BERT',
        'tipo': 'Definição',
        'pergunta': 'O que é o dataset SQuAD?',
        'resposta_esperada': 'um conjunto de dados de perguntas e respostas criado pela Stanford University para avaliação de sistemas de QA extrativo',
        'contexto': 'O SQuAD (Stanford Question Answering Dataset) é um conjunto de dados de perguntas e respostas criado pela Stanford University para avaliação de sistemas de QA extrativo. Cada pergunta do SQuAD é respondida por um trecho exato de um artigo da Wikipédia em inglês. O SQuAD tornou-se o principal benchmark para comparar sistemas de perguntas e respostas e impulsionou avanços significativos na área.'
    },
    # ── CIÊNCIA DE DADOS E MINERAÇÃO ──────────────────────────────────────
    {
        'categoria': 'Ciência de Dados',
        'tipo': 'Definição',
        'pergunta': 'O que é ciência de dados?',
        'resposta_esperada': 'uma área interdisciplinar que utiliza métodos científicos, processos, algoritmos e sistemas para extrair conhecimento e insights de dados estruturados e não estruturados',
        'contexto': 'A ciência de dados é uma área interdisciplinar que utiliza métodos científicos, processos, algoritmos e sistemas para extrair conhecimento e insights de dados estruturados e não estruturados. Combina conhecimentos de estatística, matemática, ciência da computação e conhecimento de domínio para analisar fenômenos complexos e apoiar a tomada de decisão baseada em dados.'
    },
    {
        'categoria': 'Ciência de Dados',
        'tipo': 'Definição',
        'pergunta': 'O que é mineração de dados?',
        'resposta_esperada': 'o processo de descoberta de padrões, anomalias e correlações em grandes conjuntos de dados para prever resultados',
        'contexto': 'A mineração de dados, do inglês data mining, é o processo de descoberta de padrões, anomalias e correlações em grandes conjuntos de dados para prever resultados. Usando uma ampla gama de técnicas estatísticas e de aprendizado de máquina, é possível extrair informações valiosas para aumentar receitas, reduzir custos e melhorar decisões estratégicas.'
    },
    # ── VISÃO COMPUTACIONAL E ROBÓTICA ────────────────────────────────────
    {
        'categoria': 'Visão Computacional',
        'tipo': 'Definição',
        'pergunta': 'O que é visão computacional?',
        'resposta_esperada': 'uma área da inteligência artificial que treina computadores para interpretar e compreender o mundo visual a partir de imagens e vídeos',
        'contexto': 'A visão computacional é uma área da inteligência artificial que treina computadores para interpretar e compreender o mundo visual a partir de imagens e vídeos. Utilizando técnicas de aprendizado profundo, as máquinas aprendem a identificar e classificar objetos, detectar rostos, reconhecer cenas e analisar movimentos, com aplicações em medicina, segurança, veículos autônomos e indústria.'
    },
    {
        'categoria': 'Visão Computacional',
        'tipo': 'Enumeração',
        'pergunta': 'Quais são aplicações da visão computacional?',
        'resposta_esperada': 'medicina, segurança, veículos autônomos e indústria',
        'contexto': 'A visão computacional é uma área da inteligência artificial que treina computadores para interpretar e compreender o mundo visual a partir de imagens e vídeos. Utilizando técnicas de aprendizado profundo, as máquinas aprendem a identificar e classificar objetos, detectar rostos, reconhecer cenas e analisar movimentos, com aplicações em medicina, segurança, veículos autônomos e indústria.'
    },
    {
        'categoria': 'Robótica',
        'tipo': 'Definição',
        'pergunta': 'O que é robótica?',
        'resposta_esperada': 'uma área interdisciplinar que envolve o design, construção, operação e aplicação de robôs',
        'contexto': 'A robótica é uma área interdisciplinar que envolve o design, construção, operação e aplicação de robôs. Relaciona-se com ciência da computação, eletrônica, mecânica e inteligência artificial. A robótica tem como objetivo projetar máquinas capazes de auxiliar e substituir os seres humanos em tarefas repetitivas, perigosas ou de alta precisão, com aplicações na fabricação industrial, exploração espacial, medicina e agricultura.'
    }
]

df_dataset = pd.DataFrame(dataset_qa)
print(f'Dataset de avaliacao carregado: {len(df_dataset)} pares QA')
print(f'\nDistribuicao por categoria:')
print(df_dataset['categoria'].value_counts().to_string())
print(f'\nDistribuicao por tipo de pergunta:')
print(df_dataset['tipo'].value_counts().to_string())

## 7. Métricas de Avaliação

Implementamos as métricas padrão do benchmark **SQuAD**:

- **Exact Match (EM):** 1 se a resposta predita for idêntica (após normalização) à esperada, 0 caso contrário.
- **F1 Score:** Harmônica entre precisão e revocação calculadas a nível de token.

In [ ]:
def normalize_answer(s):
    s = s.lower()
    s = ''.join(ch for ch in s if ch not in string.punctuation)
    s = ' '.join(s.split())
    return s

def get_tokens(s):
    return normalize_answer(s).split() if s else []

def compute_exact_match(pred, gold):
    return int(normalize_answer(pred) == normalize_answer(gold))

def compute_f1(pred, gold):
    pred_toks = get_tokens(pred)
    gold_toks = get_tokens(gold)
    common = Counter(pred_toks) & Counter(gold_toks)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0
    precision = num_same / len(pred_toks)
    recall = num_same / len(gold_toks)
    return (2 * precision * recall) / (precision + recall)

print('Funcoes de metrica (padrao SQuAD) definidas.')

# Teste rapido das funcoes
pred_ex  = 'John McCarthy'
gold_ex  = 'John McCarthy'
print(f'\nTeste EM (identico):  {compute_exact_match(pred_ex, gold_ex)}')
pred_ex2 = 'John'
print(f'Teste EM (parcial):   {compute_exact_match(pred_ex2, gold_ex)}')
print(f'Teste F1 (parcial):   {compute_f1(pred_ex2, gold_ex):.2f}')

In [ ]:
print('Executando avaliacao nos 30 pares QA...')
print('(Isso pode levar alguns minutos)\n')

resultados = []
for i, par in enumerate(dataset_qa):
    pred = qa_pipeline(question=par['pergunta'], context=par['contexto'])
    resposta_predita = pred['answer']
    em = compute_exact_match(resposta_predita, par['resposta_esperada'])
    f1 = compute_f1(resposta_predita, par['resposta_esperada'])
    resultados.append({
        'id': i + 1,
        'categoria': par['categoria'],
        'tipo': par['tipo'],
        'pergunta': par['pergunta'],
        'resposta_esperada': par['resposta_esperada'],
        'resposta_predita': resposta_predita,
        'score_confianca': pred['score'],
        'exact_match': em,
        'f1_score': f1
    })
    simbolo = 'OK  ' if em else ('~   ' if f1 > 0.5 else 'ERRO')
    print(f'[{simbolo}] #{i+1:02d} | F1={f1:.2f} | {par["pergunta"][:55]}')

df_resultados = pd.DataFrame(resultados)

em_medio   = df_resultados['exact_match'].mean()
f1_medio   = df_resultados['f1_score'].mean()
confianca  = df_resultados['score_confianca'].mean()

print(f'\n{"=" * 50}')
print(f'  RESULTADOS GERAIS')
print(f'{"=" * 50}')
print(f'  Exact Match medio:   {em_medio:.2%}')
print(f'  F1 Score medio:      {f1_medio:.2%}')
print(f'  Confianca media:     {confianca:.2%}')
print(f'  Total de pares:      {len(df_resultados)}')
print(f'  Acertos exatos:      {df_resultados["exact_match"].sum()}')
print(f'{"=" * 50}')

In [ ]:
# Resultados por categoria
print('\nResultados por Categoria:')
print('-' * 60)
por_categoria = df_resultados.groupby('categoria')[['exact_match', 'f1_score', 'score_confianca']].mean()
por_categoria.columns = ['EM (%)', 'F1 (%)', 'Confianca (%)']
por_categoria = por_categoria * 100
print(por_categoria.round(1).to_string())

print('\nResultados por Tipo de Pergunta:')
print('-' * 60)
por_tipo = df_resultados.groupby('tipo')[['exact_match', 'f1_score']].mean() * 100
por_tipo.columns = ['EM (%)', 'F1 (%)']
print(por_tipo.round(1).to_string())

print('\nMelhores respostas (F1 > 0.8):')
melhores = df_resultados[df_resultados['f1_score'] > 0.8][['pergunta','resposta_esperada','resposta_predita','f1_score']]
for _, row in melhores.iterrows():
    print(f'  P: {row["pergunta"][:55]}')
    print(f'  E: {row["resposta_esperada"][:55]}')
    print(f'  R: {row["resposta_predita"][:55]} (F1={row["f1_score"]:.2f})')
    print()

print('Piores respostas (F1 < 0.3):')
piores = df_resultados[df_resultados['f1_score'] < 0.3][['pergunta','resposta_esperada','resposta_predita','f1_score']]
for _, row in piores.iterrows():
    print(f'  P: {row["pergunta"][:55]}')
    print(f'  E: {row["resposta_esperada"][:55]}')
    print(f'  R: {row["resposta_predita"][:55]} (F1={row["f1_score"]:.2f})')
    print()

## 8. Visualizações dos Resultados

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Avaliacao do Sistema de QA Extrativo em Portugues', fontsize=14, fontweight='bold')

# 1. EM e F1 por categoria
cats = por_categoria.index.tolist()
x = np.arange(len(cats))
w = 0.35
axes[0,0].bar(x - w/2, por_categoria['EM (%)'], w, label='Exact Match', color='steelblue')
axes[0,0].bar(x + w/2, por_categoria['F1 (%)'], w, label='F1 Score', color='coral')
axes[0,0].set_xticks(x)
axes[0,0].set_xticklabels(cats, rotation=20, ha='right', fontsize=7)
axes[0,0].set_ylabel('Score (%)')
axes[0,0].set_title('EM e F1 por Categoria')
axes[0,0].legend()
axes[0,0].set_ylim(0, 110)

# 2. Distribuicao dos scores de confianca
axes[0,1].hist(df_resultados['score_confianca'], bins=10, color='steelblue', edgecolor='white', alpha=0.8)
axes[0,1].axvline(df_resultados['score_confianca'].mean(), color='red', linestyle='--', label=f'Media={df_resultados["score_confianca"].mean():.2f}')
axes[0,1].set_xlabel('Score de Confianca')
axes[0,1].set_ylabel('Frequencia')
axes[0,1].set_title('Distribuicao dos Scores de Confianca')
axes[0,1].legend()

# 3. F1 por tipo de pergunta
tipos = por_tipo.index.tolist()
axes[1,0].barh(tipos, por_tipo['F1 (%)'], color=sns.color_palette('muted', len(tipos)))
axes[1,0].axvline(50, color='gray', linestyle=':', alpha=0.7)
axes[1,0].set_xlabel('F1 Score (%)')
axes[1,0].set_title('F1 Score por Tipo de Pergunta')
axes[1,0].set_xlim(0, 110)
for i, v in enumerate(por_tipo['F1 (%)']):
    axes[1,0].text(v + 1, i, f'{v:.1f}%', va='center', fontsize=8)

# 4. Dispersao: Confianca vs F1
colors = df_resultados['exact_match'].map({1: 'green', 0: 'red'})
axes[1,1].scatter(df_resultados['score_confianca'], df_resultados['f1_score'],
                  c=colors, alpha=0.7, s=60, edgecolors='white')
axes[1,1].set_xlabel('Score de Confianca do Modelo')
axes[1,1].set_ylabel('F1 Score')
axes[1,1].set_title('Confianca vs F1 Score')
patch_ok  = mpatches.Patch(color='green', label='Exact Match = 1')
patch_err = mpatches.Patch(color='red',   label='Exact Match = 0')
axes[1,1].legend(handles=[patch_ok, patch_err])

plt.tight_layout()
plt.savefig('resultados_qa.png', bbox_inches='tight', dpi=150)
plt.show()
print('Figura salva em resultados_qa.png')

## 9. Conclusões do Experimento

Nesta seção, consolidamos os principais achados do sistema de QA extrativo desenvolvido.

In [ ]:
print('=' * 65)
print('  RESUMO FINAL DO EXPERIMENTO')
print('=' * 65)
print(f'  Modelo:         pierreguillou/bert-base-cased-squad-v1.1-portuguese')
print(f'  Corpus:         Wikipedia PT-BR ({len(corpus_textos)} artigos)')
print(f'  Total palavras: {len(corpus_completo.split()):,}')
print(f'  Chunks:         {len(chunks)}')
print(f'  Pares QA:       {len(df_resultados)}')
print(f'')
print(f'  Exact Match:    {df_resultados["exact_match"].mean():.2%}')
print(f'  F1 Score:       {df_resultados["f1_score"].mean():.2%}')
print(f'  Confianca:      {df_resultados["score_confianca"].mean():.2%}')
print(f'')
melhor_cat = por_categoria['F1 (%)'].idxmax()
pior_cat   = por_categoria['F1 (%)'].idxmin()
print(f'  Categoria com melhor F1: {melhor_cat} ({por_categoria.loc[melhor_cat,"F1 (%)"]:.1f}%)')
print(f'  Categoria com pior F1:   {pior_cat} ({por_categoria.loc[pior_cat,"F1 (%)"]:.1f}%)')
print('=' * 65)
print()
print('Observacoes:')
print('  - O modelo apresenta melhor desempenho em perguntas factuais curtas')
print('    (datas, nomes, siglas) do que em perguntas de definicao longa.')
print('  - O recuperador TF-IDF e eficaz para selecionar o contexto relevante,')
print('    mas pode falhar em perguntas com sinonimos ou parafraseo.')
print('  - A limitacao de 512 tokens do BERT exige truncagem em contextos longos.')
print('  - Trabalhos futuros: fine-tuning proprio em dados PT-BR, uso de')
print('    recuperadores densos (DPR) e modelos generativos (GPT, T5).')

---

## 10. Segundo Modelo: XLM-RoBERTa Multilingual

Carregamos o modelo `deepset/xlm-roberta-base-squad2`, baseado no **XLM-RoBERTa**
(Conneau et al., 2020) e ajustado no SQuAD 2.0. Diferente do BERTimbau, este
modelo é multilingual — treinado em 100 línguas simultaneamente, incluindo o
português.

**Comparação de porte:**
| Modelo | Parâmetros | Tipo | Fine-tuning |
|---|---|---|---|
| BERTimbau | ~110M | Monolingual PT | SQuAD 1.1 PT |
| XLM-RoBERTa | ~117M | Multilingual (100 línguas) | SQuAD 2.0 EN |

**Pergunta de pesquisa:** Especialização em português supera cobertura multilingual?


In [ ]:
XLM_MODEL_NAME = 'deepset/xlm-roberta-base-squad2'

print(f'Carregando segundo modelo: {XLM_MODEL_NAME}')
print('Aguarde — ~470 MB de download na primeira execucao...')

qa_pipeline_xlm = pipeline(
    'question-answering',
    model=XLM_MODEL_NAME,
    tokenizer=XLM_MODEL_NAME,
    device=device
)

print('XLM-RoBERTa carregado com sucesso!')
print(f'Dispositivo em uso: {"GPU" if device == 0 else "CPU"}')


---

## 11. Avaliação Comparativa: BERTimbau vs XLM-RoBERTa

Avaliamos o XLM-RoBERTa nos mesmos 30 pares QA usados na Seção 7,
reutilizando o `df_resultados` já gerado para o BERTimbau.


In [ ]:
# Reutilizar resultados do BERTimbau (já gerado na célula 19)
df_bert = df_resultados.copy()
df_bert['modelo'] = 'BERTimbau'

# Avaliar XLM-RoBERTa nos mesmos 30 pares
print('Executando avaliacao do XLM-RoBERTa nos 30 pares QA...')
print('(Isso pode levar alguns minutos)\n')

resultados_xlm = []
for i, par in enumerate(dataset_qa):
    pred = qa_pipeline_xlm(question=par['pergunta'], context=par['contexto'])
    resposta_predita = pred['answer']
    em = compute_exact_match(resposta_predita, par['resposta_esperada'])
    f1 = compute_f1(resposta_predita, par['resposta_esperada'])
    resultados_xlm.append({
        'id': i + 1,
        'categoria': par['categoria'],
        'tipo': par['tipo'],
        'pergunta': par['pergunta'],
        'resposta_esperada': par['resposta_esperada'],
        'resposta_predita': resposta_predita,
        'score_confianca': pred['score'],
        'exact_match': em,
        'f1_score': f1,
        'modelo': 'XLM-RoBERTa'
    })
    simbolo = 'OK  ' if em else ('~   ' if f1 > 0.5 else 'ERRO')
    print(f'[{simbolo}] #{i+1:02d} | F1={f1:.2f} | {par["pergunta"][:55]}')

df_xlm = pd.DataFrame(resultados_xlm)

# Métricas globais lado a lado
print('\n' + '=' * 60)
for nome, df_m in [('BERTimbau', df_bert), ('XLM-RoBERTa', df_xlm)]:
    print(f'  {nome}')
    print(f'    EM:         {df_m["exact_match"].mean():.2%}')
    print(f'    F1:         {df_m["f1_score"].mean():.2%}')
    print(f'    Confianca:  {df_m["score_confianca"].mean():.2%}')
    print('  ' + '-' * 40)
print('=' * 60)


In [ ]:
# Tabela global comparativa
df_comp_global = pd.DataFrame({
    'Metrica': ['Exact Match (%)', 'F1 Score (%)', 'Confianca Media (%)'],
    'BERTimbau': [
        df_bert['exact_match'].mean() * 100,
        df_bert['f1_score'].mean() * 100,
        df_bert['score_confianca'].mean() * 100
    ],
    'XLM-RoBERTa': [
        df_xlm['exact_match'].mean() * 100,
        df_xlm['f1_score'].mean() * 100,
        df_xlm['score_confianca'].mean() * 100
    ]
})
df_comp_global['Delta (XLM - BERT)'] = (
    df_comp_global['XLM-RoBERTa'] - df_comp_global['BERTimbau']
)
print('Tabela Comparativa Global:')
print(df_comp_global.round(1).to_string(index=False))

# Por categoria
por_cat_bert = df_bert.groupby('categoria')[['exact_match','f1_score']].mean() * 100
por_cat_xlm  = df_xlm.groupby('categoria')[['exact_match','f1_score']].mean() * 100
por_cat_bert.columns = ['EM_BERT', 'F1_BERT']
por_cat_xlm.columns  = ['EM_XLM',  'F1_XLM']
df_comp_cat = por_cat_bert.join(por_cat_xlm)
df_comp_cat['F1_delta'] = df_comp_cat['F1_XLM'] - df_comp_cat['F1_BERT']
print('\nComparacao por Categoria:')
print(df_comp_cat.round(1).to_string())

# Por tipo de pergunta
por_tipo_bert = df_bert.groupby('tipo')[['exact_match','f1_score']].mean() * 100
por_tipo_xlm  = df_xlm.groupby('tipo')[['exact_match','f1_score']].mean() * 100
por_tipo_bert.columns = ['EM_BERT', 'F1_BERT']
por_tipo_xlm.columns  = ['EM_XLM',  'F1_XLM']
df_comp_tipo = por_tipo_bert.join(por_tipo_xlm)
df_comp_tipo['F1_delta'] = df_comp_tipo['F1_XLM'] - df_comp_tipo['F1_BERT']
print('\nComparacao por Tipo de Pergunta:')
print(df_comp_tipo.round(1).to_string())


---

## 12. Visualizações Comparativas

Quatro gráficos comparando BERTimbau e XLM-RoBERTa:
1. F1 por categoria
2. Exact Match por tipo de pergunta
3. Confiança vs F1 (scatter)
4. Métricas globais lado a lado


In [ ]:
CORES = {'BERTimbau': 'steelblue', 'XLM-RoBERTa': 'darkorange'}

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle(
    'Comparacao: BERTimbau (monolingual PT) vs XLM-RoBERTa (multilingual)',
    fontsize=13, fontweight='bold'
)

# 1 — F1 por categoria
cats = df_comp_cat.index.tolist()
x    = np.arange(len(cats))
w    = 0.35
axes[0,0].bar(x - w/2, df_comp_cat['F1_BERT'], w,
              label='BERTimbau',   color=CORES['BERTimbau'])
axes[0,0].bar(x + w/2, df_comp_cat['F1_XLM'],  w,
              label='XLM-RoBERTa', color=CORES['XLM-RoBERTa'])
axes[0,0].set_xticks(x)
axes[0,0].set_xticklabels(cats, rotation=20, ha='right', fontsize=7)
axes[0,0].set_ylabel('F1 Score (%)')
axes[0,0].set_title('F1 por Categoria')
axes[0,0].legend(fontsize=8)
axes[0,0].set_ylim(0, 115)

# 2 — EM por tipo de pergunta
tipos = df_comp_tipo.index.tolist()
x2    = np.arange(len(tipos))
axes[0,1].bar(x2 - w/2, df_comp_tipo['EM_BERT'], w,
              label='BERTimbau',   color=CORES['BERTimbau'])
axes[0,1].bar(x2 + w/2, df_comp_tipo['EM_XLM'],  w,
              label='XLM-RoBERTa', color=CORES['XLM-RoBERTa'])
axes[0,1].set_xticks(x2)
axes[0,1].set_xticklabels(tipos, rotation=25, ha='right', fontsize=7)
axes[0,1].set_ylabel('Exact Match (%)')
axes[0,1].set_title('Exact Match por Tipo de Pergunta')
axes[0,1].legend(fontsize=8)
axes[0,1].set_ylim(0, 115)

# 3 — Scatter confiança vs F1
for nome, df_m in [('BERTimbau', df_bert), ('XLM-RoBERTa', df_xlm)]:
    axes[1,0].scatter(
        df_m['score_confianca'], df_m['f1_score'],
        label=nome, alpha=0.65, s=60,
        color=CORES[nome], edgecolors='white'
    )
axes[1,0].set_xlabel('Score de Confianca')
axes[1,0].set_ylabel('F1 Score')
axes[1,0].set_title('Confianca vs F1 (ambos os modelos)')
axes[1,0].legend(fontsize=8)

# 4 — Métricas globais
metricas  = ['Exact Match', 'F1 Score']
bert_vals = [df_bert['exact_match'].mean()*100, df_bert['f1_score'].mean()*100]
xlm_vals  = [df_xlm['exact_match'].mean()*100,  df_xlm['f1_score'].mean()*100]
xm = np.arange(len(metricas))
bars_b = axes[1,1].bar(xm - w/2, bert_vals, w,
                        label='BERTimbau',   color=CORES['BERTimbau'])
bars_x = axes[1,1].bar(xm + w/2, xlm_vals,  w,
                        label='XLM-RoBERTa', color=CORES['XLM-RoBERTa'])
axes[1,1].set_xticks(xm)
axes[1,1].set_xticklabels(metricas)
axes[1,1].set_ylabel('Score (%)')
axes[1,1].set_title('Resultados Globais — Visao Geral')
axes[1,1].legend(fontsize=8)
axes[1,1].set_ylim(0, 115)
for i, (bv, xv) in enumerate(zip(bert_vals, xlm_vals)):
    axes[1,1].text(i - w/2, bv + 1.5, f'{bv:.1f}%', ha='center', fontsize=8)
    axes[1,1].text(i + w/2, xv + 1.5, f'{xv:.1f}%', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('comparacao_modelos.png', bbox_inches='tight', dpi=150)
plt.show()
print('Figura salva em comparacao_modelos.png')


---

## 13. Análise de Concordância entre Modelos

Identificamos os pares onde os modelos concordam ou divergem no nível de
Exact Match — útil para entender os pontos fortes de cada abordagem.


In [ ]:
df_concordancia = pd.DataFrame({
    'id':             df_bert['id'].values,
    'categoria':      df_bert['categoria'].values,
    'tipo':           df_bert['tipo'].values,
    'pergunta':       df_bert['pergunta'].values,
    'resposta_esp':   df_bert['resposta_esperada'].values,
    'resp_BERT':      df_bert['resposta_predita'].values,
    'resp_XLM':       df_xlm['resposta_predita'].values,
    'em_BERT':        df_bert['exact_match'].values,
    'em_XLM':         df_xlm['exact_match'].values,
    'f1_BERT':        df_bert['f1_score'].round(3).values,
    'f1_XLM':         df_xlm['f1_score'].round(3).values,
})

ambos_certos  = df_concordancia[(df_concordancia['em_BERT']==1) & (df_concordancia['em_XLM']==1)]
ambos_errados = df_concordancia[(df_concordancia['em_BERT']==0) & (df_concordancia['em_XLM']==0)]
bert_ganha    = df_concordancia[(df_concordancia['em_BERT']==1) & (df_concordancia['em_XLM']==0)]
xlm_ganha     = df_concordancia[(df_concordancia['em_BERT']==0) & (df_concordancia['em_XLM']==1)]

total = len(df_concordancia)
print('Tabela de Concordancia (Exact Match):')
print(f'  Ambos corretos  (EM=1):   {len(ambos_certos):>3} pares  ({len(ambos_certos)/total:.0%})')
print(f'  Ambos errados   (EM=0):   {len(ambos_errados):>3} pares  ({len(ambos_errados)/total:.0%})')
print(f'  So BERTimbau correto:     {len(bert_ganha):>3} pares  ({len(bert_ganha)/total:.0%})')
print(f'  So XLM-RoBERTa correto:   {len(xlm_ganha):>3} pares  ({len(xlm_ganha)/total:.0%})')

if len(bert_ganha) > 0:
    print('\nPares onde apenas BERTimbau acertou:')
    for _, r in bert_ganha.iterrows():
        print(f'  [{r["tipo"]}] {r["pergunta"][:55]}')
        print(f'    Esperado:   {r["resposta_esp"][:55]}')
        print(f'    XLM prediu: {r["resp_XLM"][:55]}')

if len(xlm_ganha) > 0:
    print('\nPares onde apenas XLM-RoBERTa acertou:')
    for _, r in xlm_ganha.iterrows():
        print(f'  [{r["tipo"]}] {r["pergunta"][:55]}')
        print(f'    Esperado:    {r["resposta_esp"][:55]}')
        print(f'    BERT prediu: {r["resp_BERT"][:55]}')


In [ ]:
# ── Resumo final comparativo ──────────────────────────────────────────────
print('=' * 70)
print('  RESUMO FINAL COMPARATIVO')
print('=' * 70)
print(f'  {"Metrica":<28} {"BERTimbau":>12} {"XLM-RoBERTa":>13} {"Delta":>8}')
print(f'  {"-"*28} {"-"*12} {"-"*13} {"-"*8}')
for _, row in df_comp_global.iterrows():
    delta = row['Delta (XLM - BERT)']
    delta_str = f'+{delta:.1f}' if delta >= 0 else f'{delta:.1f}'
    print(f'  {row["Metrica"]:<28} {row["BERTimbau"]:>11.1f}% {row["XLM-RoBERTa"]:>12.1f}% {delta_str:>8}')
print('=' * 70)

bert_f1 = df_bert['f1_score'].mean()
xlm_f1  = df_xlm['f1_score'].mean()
if bert_f1 > xlm_f1:
    vencedor = 'BERTimbau'
    margem   = (bert_f1 - xlm_f1) * 100
    hipotese = 'especializacao monolingual em portugues supera cobertura multilingual'
else:
    vencedor = 'XLM-RoBERTa'
    margem   = (xlm_f1 - bert_f1) * 100
    hipotese = 'cobertura multilingual supera a especializacao monolingue neste corpus'
print(f'\nVencedor em F1: {vencedor} (margem: +{margem:.1f} p.p.)')
print(f'Hipotese: {hipotese}')
print('\nObservacoes:')
print('  - O SQuAD 2.0 (XLM) inclui perguntas sem resposta: o modelo pode')
print('    ser mais conservador na extracao de spans, impactando o EM.')
print('  - Perguntas Factual-Data/Pessoa (respostas curtas) tendem a')
print('    favorecer o modelo com maior especificidade lexical em PT.')
print('  - Perguntas de Definicao (respostas longas) sao avaliadas melhor')
print('    pelo F1, que permite credito parcial.')
print('\nArquivos gerados:')
print('  comparacao_modelos.png  — graficos comparativos (4 paineis)')
